# Inasistencias por seguimiento, grupo, programa y sede

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.analisis_inasistencias import enriquecer_inasistencias

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

In [ ]:
detalle = pd.read_parquet("../data/raw/inasistencias_detalle.parquet")
estudiantes = pd.read_parquet("../data/raw/estudiantes.parquet")

df = enriquecer_inasistencias(detalle, estudiantes)
print(f"Registros: {len(df):,}")
print(f"Sin seguimiento (fuera de rango): {df['Seguimiento'].isna().sum():,}")
df.head(3)

## 1. Inasistencias por grupo (A / B)

In [ ]:
por_grupo = df.groupby("Grupo").size().reset_index(name="Inasistencias")
por_grupo["%"] = (por_grupo["Inasistencias"] / por_grupo["Inasistencias"].sum() * 100).round(1).astype(str) + "%"
display(por_grupo)

sns.barplot(data=por_grupo, x="Grupo", y="Inasistencias", hue="Grupo", legend=False)
plt.title("Total de inasistencias por grupo")
plt.show()

## 2. Inasistencias por seguimiento

In [ ]:
por_seg = df.groupby("Seguimiento").size().reset_index(name="Inasistencias")
display(por_seg)

sns.barplot(data=por_seg, y="Seguimiento", x="Inasistencias", hue="Seguimiento", legend=False)
plt.title("Inasistencias por seguimiento")
plt.tight_layout()
plt.show()

## 3. Inasistencias por grupo + seguimiento

In [ ]:
por_grupo_seg = df.groupby(["Grupo", "Seguimiento"]).size().reset_index(name="Inasistencias")
display(por_grupo_seg)

g = sns.catplot(data=por_grupo_seg, x="Seguimiento", y="Inasistencias",
                col="Grupo", kind="bar", hue="Seguimiento", legend=False, sharey=False)
g.set_xticklabels(rotation=45)
plt.tight_layout()
plt.show()

## 4. Inasistencias por programa (top 15)

In [ ]:
por_programa = df.groupby("Nombre_programa_limpio").size().reset_index(name="Inasistencias").sort_values("Inasistencias", ascending=False)
display(por_programa.head(15))

top_prog = por_programa.head(15)
sns.barplot(data=top_prog, y="Nombre_programa_limpio", x="Inasistencias", hue="Nombre_programa_limpio", legend=False)
plt.title("Top 15 programas con más inasistencias")
plt.tight_layout()
plt.show()

## 5. Inasistencias por sede

In [ ]:
por_sede = df.groupby("Sede").size().reset_index(name="Inasistencias").sort_values("Inasistencias", ascending=False)
display(por_sede)

sns.barplot(data=por_sede, x="Sede", y="Inasistencias", hue="Sede", legend=False)
plt.title("Inasistencias por sede")
plt.tight_layout()
plt.show()

## 6. Tabla cruzada: Programa × Seguimiento

In [ ]:
cruzada = df.pivot_table(
    index="Nombre_programa_limpio",
    columns="Seguimiento",
    aggfunc="size",
    fill_value=0
)
display(cruzada)

plt.figure(figsize=(16, max(6, len(cruzada) * 0.4)))
sns.heatmap(cruzada, annot=True, fmt="d", cmap="Reds", linewidths=0.5)
plt.title("Inasistencias: Programa × Seguimiento")
plt.tight_layout()
plt.show()

## 7. Tabla cruzada: Sede × Seguimiento

In [ ]:
cruzada_sede = df.pivot_table(
    index="Sede",
    columns="Seguimiento",
    aggfunc="size",
    fill_value=0
)
display(cruzada_sede)

plt.figure(figsize=(12, max(4, len(cruzada_sede) * 0.5)))
sns.heatmap(cruzada_sede, annot=True, fmt="d", cmap="Blues", linewidths=0.5)
plt.title("Inasistencias: Sede × Seguimiento")
plt.tight_layout()
plt.show()

## 8. Resumen general

In [ ]:
print("=== RESUMEN ===")
print(f"Total inasistencias: {len(df):,}")
print(f"Estudiantes con inasistencias: {df['Numero_identificacion_estudiante'].nunique():,}")
print(f"Justificadas: {df['Justificacion'].sum():,} ({df['Justificacion'].mean()*100:.1f}%)")
print(f"No justificadas: {(~df['Justificacion']).sum():,} ({(~df['Justificacion']).mean()*100:.1f}%)")
print(f"Cursos involucrados: {df['Nombre_curso'].nunique()}")
print(f"Programas: {df['Nombre_programa_limpio'].nunique()}")
print(f"Sedes: {df['Sede'].nunique()}")